In [28]:
# define categories for classifier

relevant_categories = [
    "Cap On Liability", "Termination For Convenience", "Governing Law",
    "Anti-Assignment", "Change Of Control", "Renewal Term",
    "Exclusivity", "Non-Compete", "Notice Period To Terminate Renewal", "Liquidated Damages"
]

relevant_categories = [category.casefold() for category in relevant_categories]

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer

In [26]:
import import_ipynb
from chunking import tokenizer, json_to_df_new
project_root = Path.cwd().parent
train_chunks = json_to_df_new(project_root / "data/train_separate_questions.json", tokenizer)


In [27]:
train_chunks.head()

,text,start_token,end_token,char_start,char_end,num_tokens,categories
0,EXHIBIT 10.6\n\n ...,0,433,0,2867,433,"[agreement date, document name, exclusivity, l..."
1,Distributor as Company's exclusive distributor...,358,795,2166,5881,437,"[effective date, exclusivity, expiration date,..."
2,which shall commence on the date ...,720,1159,5362,8286,439,"[effective date, expiration date, minimum comm..."
3,000.00 purchase order must be received by...,1084,1499,7912,10025,415,[minimum commitment]
4,is a carry-over from that next Product Year. T...,1424,1859,9695,12388,435,[minimum commitment]


In [29]:
train_chunks = train_chunks.copy()
train_chunks = train_chunks[
    train_chunks['categories'].apply(lambda labels: any(label in relevant_categories for label in labels))
].copy()
train_chunks['categories'] = train_chunks['categories'].apply(
    lambda labels: [label for label in labels if label in relevant_categories]
)

In [30]:
# prepare training data for classifier

print(train_chunks.head())

X_train = train_chunks['text'].fillna('').astype(str)
mlb = MultiLabelBinarizer(classes=relevant_categories)
y_train = mlb.fit_transform(train_chunks['categories'])


X_train.head()

                                                 text  start_token  end_token  \
0   EXHIBIT 10.6\n\n                              ...            0        433   
1   Distributor as Company's exclusive distributor...          358        795   
2   which shall  commence on the date             ...          720       1159   
5   of or in the name of                   the oth...         1784       2226   
12  ; or\n\n                  (C)      Was   known...         4231       4665   

    char_start  char_end  num_tokens                   categories  
0            0      2867         433                [exclusivity]  
1         2166      5881         437  [exclusivity, renewal term]  
2         5362      8286         439               [renewal term]  
5        11853     15273         442                [exclusivity]  
12       30705     33821         434            [anti-assignment]  


0     EXHIBIT 10.6\n\n                              ...
1     Distributor as Company's exclusive distributor...
2     which shall  commence on the date             ...
5     of or in the name of                   the oth...
12    ; or\n\n                  (C)      Was   known...
Name: text, dtype: object

In [31]:
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)

mlb.classes_

array(['cap on liability', 'termination for convenience', 'governing law',
       'anti-assignment', 'change of control', 'renewal term',
       'exclusivity', 'non-compete', 'notice period to terminate renewal',
       'liquidated damages'], dtype=object)

In [33]:
# load test data and chunk


df_test_chunks = json_to_df_new(project_root / 'data/test.json', tokenizer)
df_test_chunks = df_test_chunks[
    df_test_chunks['categories'].apply(lambda labels: any(label in relevant_categories for label in labels))
].copy()
df_test_chunks['categories'] = df_test_chunks['categories'].apply(
    lambda labels: [label for label in labels if label in relevant_categories]
)

X_test = df_test_chunks['text'].fillna('').astype(str)
y_test = mlb.transform(df_test_chunks['categories'])

In [34]:
# Logistic Regression baseline
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear')
logreg_classifier = OneVsRestClassifier(logreg)
logreg_classifier.fit(X_train_vec, y_train)
logreg_predictions = logreg_classifier.predict(vectorizer.transform(X_test))

print('Logistic Regression multilabel results:')
print(classification_report(y_test, logreg_predictions, target_names=mlb.classes_, zero_division=0))

Logistic Regression multilabel results:
                                    precision    recall  f1-score   support

                  cap on liability       0.89      0.78      0.83       112
       termination for convenience       0.49      0.67      0.56        52
                     governing law       0.81      0.93      0.87       116
                   anti-assignment       0.78      0.79      0.78       126
                 change of control       0.66      0.48      0.56        60
                      renewal term       0.67      0.88      0.76        40
                       exclusivity       0.62      0.79      0.69        81
                       non-compete       0.67      0.74      0.70        58
notice period to terminate renewal       0.45      0.79      0.58        24
                liquidated damages       0.91      0.42      0.57        24

                         micro avg       0.71      0.76      0.73       693
                         macro avg       0.69 

In [35]:
#LinearSVC model

from sklearn.svm import LinearSVC

svc = LinearSVC(class_weight='balanced')
svc_classifier = OneVsRestClassifier(svc)
svc_classifier.fit(X_train_vec, y_train)
svc_predictions = svc_classifier.predict(vectorizer.transform(X_test))

print('LinearSVC multilabel results:')
print(classification_report(y_test, svc_predictions, target_names=mlb.classes_, zero_division=0))


LinearSVC multilabel results:
                                    precision    recall  f1-score   support

                  cap on liability       0.91      0.77      0.83       112
       termination for convenience       0.62      0.62      0.62        52
                     governing law       0.96      0.92      0.94       116
                   anti-assignment       0.82      0.71      0.76       126
                 change of control       0.96      0.37      0.53        60
                      renewal term       0.83      0.75      0.79        40
                       exclusivity       0.70      0.67      0.68        81
                       non-compete       0.87      0.59      0.70        58
notice period to terminate renewal       0.64      0.58      0.61        24
                liquidated damages       1.00      0.33      0.50        24

                         micro avg       0.83      0.69      0.75       693
                         macro avg       0.83      0.63 